In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
import psycopg2
import pandas as pd
from datetime import date, timedelta

In [4]:
print(os.getenv("DB_HOST"))

aws-0-ap-south-1.pooler.supabase.com


In [51]:
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_PORT = os.getenv("DB_PORT", "5432")

In [52]:
#Connect to DB
conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    port=DB_PORT
)

In [53]:
cur = conn.cursor()

In [42]:
#Pull data from SQL 
orders_df = pd.read_sql("""
    SELECT customer_id, created_at, total
    FROM ecom.orders
    WHERE status != 'cancelled' """, conn)

C:\Users\MonalisaPal\AppData\Local\Temp\ipykernel_21372\4108203494.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders_df = pd.read_sql("""


In [147]:
orders_df.head()

,customer_id,created_at,total
0,797,2025-09-28 11:02:25+00:00,14825.13
1,2228,2025-09-28 11:15:09+00:00,6466.64
2,6388,2025-09-28 11:16:56+00:00,1651.47
3,2120,2025-09-28 11:19:54+00:00,15037.90
4,306,2025-09-28 11:27:34+00:00,1145.77


In [8]:
run_date = pd.to_datetime(date.today()) #converting to timestamp

last_90_days = run_date - timedelta(days=90)

orders_df["created_at"] = (pd.to_datetime(orders_df["created_at"]).dt.tz_localize(None)) #Also make it time zone-naive just like run_date

In [9]:
#Recency: days since last completed order (latest order date) --- max(order_date)
recency_df = (orders_df
        .groupby("customer_id", as_index=False)
        .agg(last_order_date=("created_at", "max"))
)

In [10]:
recency_df.head()

,customer_id,last_order_date
0,1,2025-12-07 10:57:26
1,2,2025-12-18 09:56:15
2,3,2025-10-26 02:38:06
3,4,2025-10-31 04:38:28
4,5,2025-12-20 09:40:27


In [11]:
recency_df["recency_days"] = (run_date - recency_df["last_order_date"]).dt.days

In [12]:
#Frequency: number of orders in the last 90 days

last_90_days_orders = orders_df[orders_df["created_at"] >= last_90_days]
last_90_days_orders = (last_90_days_orders.dropna(subset=["customer_id", "total"]))

frequency_df = (last_90_days_orders
    .groupby("customer_id", as_index=False)
    .agg(frequency_orders=("created_at", "size"), monetary_value=("total", "sum"))
)

In [13]:
#RFM metric
rfm_df = (recency_df.merge(frequency_df, on="customer_id", how="left")
        .fillna({"frequency_orders": 0,"monetary_value": 0}))

rfm_df["frequency_orders"] = rfm_df["frequency_orders"].astype(int) #for NaN values, pandas upcasts coln to float so need to change frequent_orders to int again

In [14]:
rfm_df.head(3)

,customer_id,last_order_date,recency_days,frequency_orders,monetary_value
0,1,2025-12-07 10:57:26,93,0,0.00
1,2,2025-12-18 09:56:15,82,1,8702.46
2,3,2025-10-26 02:38:06,135,0,0.00


In [15]:
#Create scores (1–5) for R, F, M using quantiles (or percentile buckets)

#for recency days, lower the value better the score
rfm_df["r_rank"] = rfm_df["recency_days"].rank(method="first", ascending=True)
rfm_df["r_score"] = pd.cut(rfm_df["r_rank"], bins=5,labels=[5, 4, 3, 2, 1]).astype(int) 

In [16]:
#If freq higher then better
rfm_df["f_rank"] = rfm_df["frequency_orders"].rank(method="first", ascending=False)
rfm_df["f_score"] = pd.cut(rfm_df["f_rank"],bins=5,labels=[5, 4, 3, 2, 1]).astype(int)

In [17]:
#Similarly higher monetary value --> better score
rfm_df["m_rank"] = rfm_df["monetary_value"].rank(method="first", ascending=False)
rfm_df["m_score"] = pd.cut(rfm_df["m_rank"],bins=5,labels=[5, 4, 3, 2, 1]).astype(int)

In [18]:
#RFM segmentation - assigning labels
def assign_labels(row):
    if row.r_score >= 4 and row.f_score >= 4 and row.m_score >= 4:
        return "Champions"
    elif row.f_score >= 4 and row.r_score >= 3:
        return "Loyal"
    elif row.m_score >= 4 and row.f_score <= 3:
        return "Big Spenders"
    elif row.r_score <= 2 and row.f_score >= 3:
        return "At Risk"
    elif row.r_score <= 2 and row.f_score <= 2:
        return "Hibernating"
    else:
        return "Others"

rfm_df["rfm_segment"] = rfm_df.apply(assign_labels, axis=1)

In [22]:
rfm_df["run_date"] = run_date

In [23]:
rfm_df["rfm_score"] = (rfm_df["r_score"].astype(str) + rfm_df["f_score"].astype(str) + rfm_df["m_score"].astype(str))

In [24]:
final_cols = [
    "run_date",
    "customer_id",
    "recency_days",
    "frequency_orders",
    "monetary_value",
    "r_score",
    "f_score",
    "m_score",
    "rfm_score",
    "rfm_segment",
]

final_df = rfm_df[final_cols]

In [25]:
final_df.head()

,run_date,customer_id,recency_days,frequency_orders,monetary_value,r_score,f_score,m_score,rfm_score,rfm_segment
0,2026-03-11,1,93,0,0.00,3,3,3,333,Others
1,2026-03-11,2,82,1,8702.46,4,5,4,454,Champions
2,2026-03-11,3,135,0,0.00,1,3,3,133,At Risk
3,2026-03-11,4,130,0,0.00,1,3,3,133,At Risk
4,2026-03-11,5,80,1,500.61,4,5,3,453,Loyal


In [122]:
pd.read_sql("""
    SELECT *
    FROM ecom.customer_rfm_daily limit 2 """, conn)

C:\Users\MonalisaPal\AppData\Local\Temp\ipykernel_7544\3170633895.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""


,run_date,customer_id,recency_days,frequency_orders,monetary_value,r_score,f_score,m_score,rfm_score,rfm_segment,comments,created_at,updated_at,logs
0,2026-02-25,12,61,1,11767.60,5,1,3,9,Others,None,2026-02-25 18:07:16.345129,2026-02-25 14:27:43.783625,{'source': 'daily_rfm_pipeline'}
1,2026-02-25,13,83,1,10640.22,1,1,3,5,Hibernating,None,2026-02-25 18:07:16.345129,2026-02-25 14:27:43.783625,{'source': 'daily_rfm_pipeline'}


In [26]:
upsert_sql = """
INSERT INTO ecom.customer_rfm_daily (
    run_date,
    customer_id,
    recency_days,
    frequency_orders,
    monetary_value,
    r_score,
    f_score,
    m_score,
    rfm_score,
    rfm_segment
)
VALUES (
    %(run_date)s,
    %(customer_id)s,
    %(recency_days)s,
    %(frequency_orders)s,
    %(monetary_value)s,
    %(r_score)s,
    %(f_score)s,
    %(m_score)s,
    %(rfm_score)s,
    %(rfm_segment)s
)
ON CONFLICT (run_date, customer_id)
DO UPDATE SET 
    recency_days = EXCLUDED.recency_days,
    frequency_orders = EXCLUDED.frequency_orders,
    monetary_value = EXCLUDED.monetary_value,
    r_score = EXCLUDED.r_score,
    f_score = EXCLUDED.f_score,
    m_score = EXCLUDED.m_score,
    rfm_score = EXCLUDED.rfm_score,
    rfm_segment = EXCLUDED.rfm_segment,
    updated_at = CURRENT_TIMESTAMP;
"""

##If new row insertion fails due to conflict, then exclude the new row, instead update with the new row's(excluded rows') values

In [89]:
records_dict = final_df.to_dict(orient="records")

In [134]:
final_df.shape

(9789, 10)

In [58]:
first_row_dict = final_df.iloc[1].to_dict()

In [59]:
first_row_dict["run_date"] = first_row_dict["run_date"].strftime("%B %d, %Y")

In [37]:
cur.execute(upsert_sql, first_row_dict)
conn.commit()